# OpenPlaque End-to-End Segmentation + Boundary Refinement

Fresh Colab workflow:

1. Mount Google Drive
2. Clone/update OpenPlaque from GitHub
3. Install requirements
4. Configure nnU-Net
5. Load full DICOM study
6. Segment LAD/RCA/LCX curved reformats
7. Compute raw TPV
8. Apply experimental boundary refinement
9. Compute refined TPV
10. Save masks and summary

Expected Drive files:

```text
/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip
/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip
```

Research use only. Not for clinical decision-making.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Clone or update OpenPlaque
!git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque || true
!git -C /content/OpenPlaque pull


In [ ]:
# Install requirements
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt


In [ ]:
# Configure Python path and nnU-Net environment
import os
import sys
from pathlib import Path

repo = Path("/content/OpenPlaque")
src = repo / "src"

if str(src) not in sys.path:
    sys.path.insert(0, str(src))

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

for d in [os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]]:
    os.makedirs(d, exist_ok=True)

print("OpenPlaque path:", src)
print("nnUNet_results:", os.environ["nnUNet_results"])


In [ ]:
# Verify GPU
!nvidia-smi


In [ ]:
# Extract nnU-Net model weights if needed
import zipfile
from pathlib import Path

model_zip = Path("/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip")
model_target = Path("/content/nnUNet_results/Dataset001_CCTA_DHM")

if model_target.exists():
    print("Model already extracted:", model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f"Missing model ZIP: {model_zip}")
    print("Extracting model ZIP...")
    with zipfile.ZipFile(model_zip) as z:
        z.extractall("/content/nnUNet_results")
    print("Extracted model to /content/nnUNet_results")

!find /content/nnUNet_results -maxdepth 3


In [ ]:
# Load full DICOM study
from openplaque.study import OpenPlaqueStudy

study_zip = "/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip"
study = OpenPlaqueStudy(study_zip)

study.summary()


## Automatically detect curved coronary reformats

The workflow now tries to identify LAD/RCA/LCX curved reformats from DICOM series metadata.
For this UCLA study, the known series numbers are kept only as a fallback.


In [ ]:
# Automatically detect LAD/RCA/LCX curved reformats
# Fallback values are used only if metadata is unavailable or ambiguous.
fallback_series = {"RCA": 1035, "LCX": 1039, "LAD": 1043}
series_map = detect_artery_series(study, fallback_series=fallback_series)
print("Detected artery series:", series_map)

image_lad, volume_lad, _ = study.load_series(series_map["LAD"])
image_rca, volume_rca, _ = study.load_series(series_map["RCA"])
image_lcx, volume_lcx, _ = study.load_series(series_map["LCX"])

print("LAD:", volume_lad.shape, image_lad.GetSpacing())
print("RCA:", volume_rca.shape, image_rca.GetSpacing())
print("LCX:", volume_lcx.shape, image_lcx.GetSpacing())


In [ ]:
# Quick visual check
import matplotlib.pyplot as plt

def show_mid(volume, title):
    z = volume.shape[0] // 2
    plt.figure(figsize=(6,6))
    plt.imshow(volume[z], cmap="gray", vmin=-200, vmax=800)
    plt.title(title)
    plt.axis("off")
    plt.show()

show_mid(volume_lad, "LAD curved reformat")
show_mid(volume_rca, "RCA curved reformat")
show_mid(volume_lcx, "LCX curved reformat")


In [ ]:
# Import OpenPlaque modules
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series
from openplaque.uncertainty import make_tpv_uncertainty_summary
from openplaque.report import write_html_report


## Run plaque segmentation

This runs the pretrained `Dataset001_CCTA_DHM` nnU-Net model on each curved artery reformat.


In [ ]:
lad_report = segment_vessel(image_lad, volume_lad, "LAD")
lad_report.summary()
lad_report.show_overlay(label=2)


In [ ]:
# Automatically detect LAD/RCA/LCX curved reformats
# Fallback values are used only if metadata is unavailable or ambiguous.
fallback_series = {"RCA": 1035, "LCX": 1039, "LAD": 1043}
series_map = detect_artery_series(study, fallback_series=fallback_series)
print("Detected artery series:", series_map)

image_lad, volume_lad, _ = study.load_series(series_map["LAD"])
image_rca, volume_rca, _ = study.load_series(series_map["RCA"])
image_lcx, volume_lcx, _ = study.load_series(series_map["LCX"])

print("LAD:", volume_lad.shape, image_lad.GetSpacing())
print("RCA:", volume_rca.shape, image_rca.GetSpacing())
print("LCX:", volume_lcx.shape, image_lcx.GetSpacing())


In [ ]:
lcx_report = segment_vessel(image_lcx, volume_lcx, "LCX")
lcx_report.summary()
lcx_report.show_overlay(label=2)


## Raw total plaque volume

In [ ]:
reports = [lad_report, rca_report, lcx_report]

raw_total_tpv = sum(r.tpv_mm3 for r in reports)

print("Raw plaque volume by vessel")
print("-" * 40)

for r in reports:
    print(f"{r.name}: {r.tpv_mm3:.2f} mm³ ({r.plaque_voxels} voxels)")

print("-" * 40)
print(f"RAW TOTAL PLAQUE VOLUME: {raw_total_tpv:.2f} mm³")


## Boundary refinement

Default experimental refinement:

- Remove tiny disconnected plaque components
- Remove plaque voxels adjacent to the predicted normal-vessel label
- Do **not** erode plaque core by default
- Do **not** use HU thresholds by default


In [ ]:
def refine_report(report, min_component_voxels=10, lumen_distance_voxels=1):
    return refine_plaque_mask(
        volume=report.volume,
        mask=report.mask,
        spacing=report.mask_image.GetSpacing(),
        remove_small=True,
        min_component_voxels=min_component_voxels,
        trim_lumen_adjacent=True,
        lumen_distance_voxels=lumen_distance_voxels,
        erode_core=False,
        high_hu_threshold=None,
        low_hu_threshold=None,
    )

lad_refined = refine_report(lad_report)
rca_refined = refine_report(rca_report)
lcx_refined = refine_report(lcx_report)

for name, ref in [("LAD", lad_refined), ("RCA", rca_refined), ("LCX", lcx_refined)]:
    print("\n" + "=" * 40)
    print(name)
    ref.summary()


In [ ]:
# Show refined overlays
lad_refined.show_refined_overlay(lad_report.volume)
rca_refined.show_refined_overlay(rca_report.volume)
lcx_refined.show_refined_overlay(lcx_report.volume)


In [ ]:
# Show removed boundary voxels
lad_refined.show_removed_overlay(lad_report.volume)
rca_refined.show_removed_overlay(rca_report.volume)
lcx_refined.show_removed_overlay(lcx_report.volume)


## Raw vs refined TPV summary

In [ ]:
refinements = {
    "LAD": lad_refined,
    "RCA": rca_refined,
    "LCX": lcx_refined,
}

refined_total_tpv = sum(r.refined_tpv_mm3 for r in refinements.values())
removed_total = sum(r.removed_volume_mm3 for r in refinements.values())

print("Raw vs refined plaque volume")
print("-" * 65)
print(f"{'Vessel':<6} {'Raw TPV':>12} {'Refined TPV':>14} {'Removed':>12}")
print("-" * 65)

for report in reports:
    ref = refinements[report.name]
    print(f"{report.name:<6} {report.tpv_mm3:12.2f} {ref.refined_tpv_mm3:14.2f} {ref.removed_volume_mm3:12.2f}")

print("-" * 65)
print(f"{'TOTAL':<6} {raw_total_tpv:12.2f} {refined_total_tpv:14.2f} {removed_total:12.2f}")


## Optional: high-confidence plaque core

This is more conservative and will likely underestimate plaque volume, but it helps estimate boundary uncertainty.


In [ ]:
core_results = {}

for report in reports:
    core_results[report.name] = refine_plaque_mask(
        volume=report.volume,
        mask=report.mask,
        spacing=report.mask_image.GetSpacing(),
        remove_small=True,
        min_component_voxels=10,
        trim_lumen_adjacent=True,
        lumen_distance_voxels=1,
        erode_core=True,
        erosion_iterations=1,
    )

print("High-confidence core plaque volume")
print("-" * 50)

core_total = 0
for name, core in core_results.items():
    core_total += core.refined_tpv_mm3
    print(f"{name}: {core.refined_tpv_mm3:.2f} mm³")

print("-" * 50)
print(f"CORE TOTAL: {core_total:.2f} mm³")


## TPV uncertainty interval

Reports high-confidence core TPV to raw AI TPV as an uncertainty interval, with refined TPV as the working estimate.


In [ ]:
uncertainty = make_tpv_uncertainty_summary(reports, refinements, core_results)

print("TPV uncertainty interval")
print("-" * 90)
print(f"{'Vessel':<8} {'Core':>12} {'Refined':>12} {'Raw':>12} {'Interval':>22} {'Width':>12}")
print("-" * 90)
for row in uncertainty.rows():
    interval = f"{row['interval_low_mm3']:.2f} – {row['interval_high_mm3']:.2f}"
    print(f"{row['vessel']:<8} {row['core_tpv_mm3']:12.2f} {row['refined_tpv_mm3']:12.2f} {row['raw_tpv_mm3']:12.2f} {interval:>22} {row['uncertainty_width_mm3']:12.2f}")


## Generate HTML report

Creates a self-contained HTML report with TPV tables and raw/refined/core overlay images.


In [ ]:
report_path = "/content/drive/MyDrive/OpenPlaque/Segmentations/openplaque_tpv_boundary_report.html"
write_html_report(
    output_html=report_path,
    reports=reports,
    refined_results=refinements,
    core_results=core_results,
    uncertainty_summary=uncertainty,
)
print("Saved HTML report:", report_path)


## Save outputs to Google Drive

In [ ]:
import os
import SimpleITK as sitk

save_dir = "/content/drive/MyDrive/OpenPlaque/Segmentations"
os.makedirs(save_dir, exist_ok=True)

# Save raw masks
lad_report.save_mask(f"{save_dir}/LAD_raw_plaque_segmentation.nii.gz")
rca_report.save_mask(f"{save_dir}/RCA_raw_plaque_segmentation.nii.gz")
lcx_report.save_mask(f"{save_dir}/LCX_raw_plaque_segmentation.nii.gz")

# Save refined masks as NIfTI
def save_refined_mask(ref, reference_image, path):
    img = sitk.GetImageFromArray(ref.refined_mask.astype("uint8"))
    img.CopyInformation(reference_image)
    sitk.WriteImage(img, path)
    print("Saved:", path)

save_refined_mask(lad_refined, lad_report.mask_image, f"{save_dir}/LAD_refined_plaque_segmentation.nii.gz")
save_refined_mask(rca_refined, rca_report.mask_image, f"{save_dir}/RCA_refined_plaque_segmentation.nii.gz")
save_refined_mask(lcx_refined, lcx_report.mask_image, f"{save_dir}/LCX_refined_plaque_segmentation.nii.gz")


In [ ]:
# Save text summary
summary_path = "/content/drive/MyDrive/OpenPlaque/Segmentations/tpv_boundary_refinement_summary.txt"

with open(summary_path, "w") as f:
    f.write("OpenPlaque experimental plaque segmentation + boundary refinement summary\n")
    f.write("Research use only. Not for clinical decision-making.\n\n")
    f.write("Raw vs refined plaque volume\n")
    f.write("---------------------------------------------\n")
    for report in reports:
        ref = refinements[report.name]
        f.write(f"{report.name}: raw={report.tpv_mm3:.2f} mm3, refined={ref.refined_tpv_mm3:.2f} mm3, removed={ref.removed_volume_mm3:.2f} mm3\n")
    f.write("---------------------------------------------\n")
    f.write(f"RAW TOTAL TPV: {raw_total_tpv:.2f} mm3\n")
    f.write(f"REFINED TOTAL TPV: {refined_total_tpv:.2f} mm3\n")
    f.write(f"REMOVED BOUNDARY VOLUME: {removed_total:.2f} mm3\n")
    f.write(f"CORE TOTAL TPV: {core_total:.2f} mm3\n")

print("Saved:", summary_path)
